# 01 — SQL: fundamentos y análisis de datos

Consultas sencillas hasta análisis con varias tablas y funciones de ventana.

**TPC-DS**: dataset que simula escenarios de retail y comercio electrónico.

Se empezar con consultas básicas y se aumenta la complejidad.

## 1. Tablas

Revisando tablas disponibles en TPC-DS.

In [0]:
%sql
SHOW TABLES IN samples.tpcds_sf1;

Revisar estructura de algunas tablas.

In [0]:
%sql
DESCRIBE TABLE samples.tpcds_sf1.customer;

In [0]:
%sql
DESCRIBE TABLE samples.tpcds_sf1.store_sales;

## 2. Primeras consultas


In [0]:
%sql
SELECT *
FROM samples.tpcds_sf1.customer
LIMIT 5;

In [0]:
%sql
SELECT COUNT(*) AS total_clientes
FROM samples.tpcds_sf1.customer;

Seleccion de columnas

In [0]:
%sql
SELECT
    c_customer_sk,
    c_first_name,
    c_last_name,
    c_birth_country
FROM samples.tpcds_sf1.customer
WHERE c_birth_country = 'UNITED STATES'
LIMIT 5;

## 3. Agregaciones

Revisar registros individuales, obtener algunos indicadores por categoría.

In [0]:
%sql
SELECT
    i_category,
    COUNT(*) AS cantidad_productos,
    ROUND(AVG(i_current_price), 2) AS precio_promedio,
    ROUND(MIN(i_current_price), 2) AS precio_minimo,
    ROUND(MAX(i_current_price), 2) AS precio_maximo
FROM samples.tpcds_sf1.item
GROUP BY i_category
ORDER BY precio_promedio DESC


## 4. Ordenar y crear categorías


In [0]:
%sql
SELECT
    i_item_sk,
    i_item_id,
    i_category,
    i_current_price,
    CASE
        WHEN i_current_price < 35 THEN 'Bajo'
        WHEN i_current_price < 70 THEN 'Medio'
        ELSE 'Alto'
    END AS rango_precio
FROM samples.tpcds_sf1.item
ORDER BY i_current_price DESC
LIMIT 5;

**HAVING**: filtrar despues de realizar una agregación.

In [0]:
%sql
SELECT
    i_category,
    COUNT(*) AS cantidad_productos
FROM samples.tpcds_sf1.item
GROUP BY i_category
HAVING COUNT(*) <= 1000
ORDER BY cantidad_productos DESC;

## 5. Fechas

Tabla **date_dim** contiene información para analizar los datos por año, mes o trimestre.

In [0]:
%sql
SELECT
    d_date,
    d_year,
    d_moy,
    d_qoy,
    d_week_seq
FROM samples.tpcds_sf1.date_dim
WHERE d_year BETWEEN 2000 AND 2002
ORDER BY d_date
LIMIT 10;

## 6. JOIN

Relacionar las ventas con los productos para conocer los ingresos por categoría.

In [0]:
%sql
SELECT
    i.i_category,
    SUM(ss.ss_net_paid) AS ingresos
FROM samples.tpcds_sf1.store_sales AS ss
INNER JOIN samples.tpcds_sf1.item AS i
    ON ss.ss_item_sk = i.i_item_sk
GROUP BY i.i_category
ORDER BY ingresos DESC;

Incorporar la fecha para analizar el resultado por año.

In [0]:
%sql
SELECT
    d.d_year,
    i.i_category,
    ROUND(SUM(ss.ss_net_paid), 2) AS ingresos
FROM samples.tpcds_sf1.store_sales AS ss
INNER JOIN samples.tpcds_sf1.item AS i
    ON ss.ss_item_sk = i.i_item_sk
INNER JOIN samples.tpcds_sf1.date_dim AS d
    ON ss.ss_sold_date_sk = d.d_date_sk
GROUP BY
    d.d_year,
    i.i_category
ORDER BY
    d.d_year,
    ingresos DESC;

## 7. CTE

Common Table Expression (CTE) para separar los pasos y que sea mas sencillo de leer.

In [0]:
%sql
WITH ingresos_categoria AS (
    SELECT
        i.i_category,
        SUM(ss.ss_net_paid) AS ingresos
    FROM samples.tpcds_sf1.store_sales AS ss
    INNER JOIN samples.tpcds_sf1.item AS i
        ON ss.ss_item_sk = i.i_item_sk
    GROUP BY i.i_category
)

SELECT
    i_category,
    ROUND(ingresos, 0) AS ingresos
FROM ingresos_categoria
WHERE ingresos > 450000000
ORDER BY ingresos DESC;

## 8. Subconsultas

Para buscar los productos que tienen un precio por encima del promedio.

In [0]:
%sql
SELECT
    i_item_id,
    i_class,
    i_category,
    i_current_price
FROM samples.tpcds_sf1.item
WHERE i_current_price > (
    SELECT AVG(i_current_price)
    FROM samples.tpcds_sf1.item
)
ORDER BY i_current_price DESC
LIMIT 5;

## 9. Funciones de ventana

Comparar registros entre sí sin perder el detalle de cada fila.

Primero ordena las categorías según sus ingresos dentro de cada año.

In [0]:
%sql
WITH ingresos_categoria AS (
    SELECT
        d.d_year,
        i.i_category,
        SUM(ss.ss_net_paid) AS ingresos
    FROM samples.tpcds_sf1.store_sales AS ss
    INNER JOIN samples.tpcds_sf1.item AS i
        ON ss.ss_item_sk = i.i_item_sk
    INNER JOIN samples.tpcds_sf1.date_dim AS d
        ON ss.ss_sold_date_sk = d.d_date_sk
    GROUP BY
        d.d_year,
        i.i_category
)

SELECT
    d_year,
    i_category,
    ROUND(ingresos, 0) AS ingresos,
    RANK() OVER (
        PARTITION BY d_year
        ORDER BY ingresos DESC
    ) AS ranking_anual
FROM ingresos_categoria
ORDER BY
    d_year,
    ranking_anual;

Otra función de ventana para calcular un acumulado dentro de cada año.

In [0]:
%sql
WITH ingresos_categoria AS (
    SELECT
        d.d_year,
        i.i_category,
        SUM(ss.ss_net_paid) AS ingresos
    FROM samples.tpcds_sf1.store_sales AS ss
    INNER JOIN samples.tpcds_sf1.item AS i
        ON ss.ss_item_sk = i.i_item_sk
    INNER JOIN samples.tpcds_sf1.date_dim AS d
        ON ss.ss_sold_date_sk = d.d_date_sk
    GROUP BY
        d.d_year,
        i.i_category
)

SELECT
    d_year,
    i_category,
    ROUND(ingresos, 0) AS ingresos,
    ROUND(
        SUM(ingresos) OVER (
            PARTITION BY d_year
            ORDER BY ingresos DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),0) AS ingresos_acumulados
FROM ingresos_categoria
ORDER BY
    d_year,
    ingresos DESC;

## 10. Participación dentro del año

Funciones de ventana para calcular qué porcentaje de ingresos anuales corresponde a cada categoría.

In [0]:
%sql
WITH ingresos_categoria AS (
    SELECT
        d.d_year,
        i.i_category,
        SUM(ss.ss_net_paid) AS ingresos
    FROM samples.tpcds_sf1.store_sales AS ss
    INNER JOIN samples.tpcds_sf1.item AS i
        ON ss.ss_item_sk = i.i_item_sk
    INNER JOIN samples.tpcds_sf1.date_dim AS d
        ON ss.ss_sold_date_sk = d.d_date_sk
    GROUP BY
        d.d_year,
        i.i_category
)

SELECT
    d_year,
    i_category,
    ROUND(ingresos, 0) AS ingresos,
    ROUND(
        100 * ingresos / SUM(ingresos) OVER (
            PARTITION BY d_year
        ),2 ) AS participacion_anual_pct
FROM ingresos_categoria
ORDER BY
    d_year,
    participacion_anual_pct DESC;

## 11. Análisis por cliente

Relacionar los clientes con las ventas para obtener indicadores individuales.

In [0]:
%sql
SELECT
    c.c_customer_sk,
    c.c_first_name,
    c.c_last_name,
    ROUND(SUM(ss.ss_net_paid), 0) AS total_compras
FROM samples.tpcds_sf1.customer AS c
INNER JOIN samples.tpcds_sf1.store_sales AS ss
    ON c.c_customer_sk = ss.ss_customer_sk
GROUP BY
    c.c_customer_sk,
    c.c_first_name,
    c.c_last_name
ORDER BY total_compras DESC
LIMIT 5;

## 12. Consulta final

Identificar, para cada año, las tres categorías con mayores ingresos y mostrar qué porcentaje representan sobre el total anual.

In [0]:
%sql
WITH ingresos_categoria AS (
    SELECT
        d.d_year,
        i.i_category,
        SUM(ss.ss_net_paid) AS ingresos
    FROM samples.tpcds_sf1.store_sales AS ss
    INNER JOIN samples.tpcds_sf1.item AS i
        ON ss.ss_item_sk = i.i_item_sk
    INNER JOIN samples.tpcds_sf1.date_dim AS d
        ON ss.ss_sold_date_sk = d.d_date_sk
    GROUP BY
        d.d_year,
        i.i_category
),

ranking AS (
    SELECT
        d_year,
        i_category,
        ingresos,
        RANK() OVER (
            PARTITION BY d_year
            ORDER BY ingresos DESC
        ) AS posicion
    FROM ingresos_categoria
)

SELECT
    d_year,
    i_category,
    ROUND(ingresos, 0) AS ingresos,
    posicion,
    ROUND(
        100 * ingresos / SUM(ingresos) OVER (
            PARTITION BY d_year
        ), 2) AS participacion_anual_pct
FROM ranking
WHERE posicion <= 3
ORDER BY
    d_year,
    posicion;